In [ ]:
"""
RQ2 — Comparative SHAP Analysis
==================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption
 
PURPOSE
-------
Directly answers RQ2: "Do SHAP feature importance profiles differ between the
revenue growth model and the cost reduction model?"
 
Loads the SHAP values already computed and saved by shared_pipeline.py for
BOTH targets, then produces:
  1. A merged comparison table — mean |SHAP| (global importance) and mean
     signed SHAP (direction) for every feature, side by side for both targets
  2. A horizontal diverging bar chart — the classic "tornado" comparison plot,
     showing each feature's importance for revenue growth (left) vs cost
     reduction (right)
  3. A rank-correlation summary — quantifies how similar/different the two
     importance rankings are overall (Spearman correlation between ranks)
  4. A "top divergent features" table — features that matter a lot for ONE
     target but barely at all for the other (the most interesting findings
     for your Discussion chapter)
 
Run: python comparative_shap_analysis.py
Requires: shared_pipeline.py must have been run for BOTH targets first,
          so that pipeline_outputs/shap/revenue_growth_percent_shap_values.pkl
          and pipeline_outputs/shap/cost_reduction_percent_shap_values.pkl
          both exist.
"""
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from scipy.stats import spearmanr
 
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
 
SHAP_DIR = "pipeline_outputs/shap"
OUT_DIR = "pipeline_outputs/comparative_shap"
os.makedirs(OUT_DIR, exist_ok=True)
 
TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
 
TARGET_1_LABEL = "Revenue Growth"
TARGET_2_LABEL = "Cost Reduction"
 
NAVY = "#1F4E79"
AMBER = "#E8A33D"
GREY = "#909497"
 
TOP_N_DISPLAY = 15   # how many features to show in the tornado plot
TOP_N_DIVERGENT = 10  # how many "most divergent" features to report
 
# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BOTH SAVED SHAP FILES
# ═══════════════════════════════════════════════════════════════════════════
 
def load_shap_data(target_column, shap_dir=SHAP_DIR):
    path = f"{shap_dir}/{target_column}_shap_values.pkl"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nCould not find '{path}'.\n"
            f"Make sure shared_pipeline.py has been run for target_column="
            f"'{target_column}' first (this file is created automatically "
            f"by the pipeline's SHAP computation step)."
        )
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data
 
print("=" * 80)
print("LOADING SAVED SHAP VALUES FOR BOTH TARGETS")
print("=" * 80)
 
data_1 = load_shap_data(TARGET_1)
data_2 = load_shap_data(TARGET_2)
 
print(f"✅ Loaded {TARGET_1}: {data_1['shap_values'].shape[0]:,} observations, "
      f"{len(data_1['feature_names'])} features")
print(f"✅ Loaded {TARGET_2}: {data_2['shap_values'].shape[0]:,} observations, "
      f"{len(data_2['feature_names'])} features")
 
# ═══════════════════════════════════════════════════════════════════════════
# 2. BUILD PER-FEATURE SUMMARY FOR EACH TARGET
# ═══════════════════════════════════════════════════════════════════════════
# Global importance = mean(|SHAP value|) across all test observations
# Direction        = mean(SHAP value) — positive means the feature tends to
#                    push predictions UP, negative means it tends to push
#                    predictions DOWN, on average across the test set
 
def summarise_shap(shap_values, feature_names):
    shap_values = np.asarray(shap_values)
    mean_abs = np.abs(shap_values).mean(axis=0)
    mean_signed = shap_values.mean(axis=0)
    df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs,
        "mean_signed_shap": mean_signed,
    })
    df["rank"] = df["mean_abs_shap"].rank(ascending=False, method="min").astype(int)
    return df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
 
summary_1 = summarise_shap(data_1["shap_values"], data_1["feature_names"])
summary_2 = summarise_shap(data_2["shap_values"], data_2["feature_names"])
 
print(f"\nTop 5 features for {TARGET_1_LABEL}:")
print(summary_1.head(5).to_string(index=False))
print(f"\nTop 5 features for {TARGET_2_LABEL}:")
print(summary_2.head(5).to_string(index=False))
 
# ═══════════════════════════════════════════════════════════════════════════
# 3. MERGE INTO A SINGLE COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════
# Outer merge — some features may exist in one model's one-hot encoding but
# not the other (e.g. if a rare category only appears in one target's data
# after row-dropping). Missing values are filled with 0 (no importance).
 
comparison = pd.merge(
    summary_1[["feature", "mean_abs_shap", "mean_signed_shap", "rank"]],
    summary_2[["feature", "mean_abs_shap", "mean_signed_shap", "rank"]],
    on="feature", how="outer", suffixes=(f"_{TARGET_1}", f"_{TARGET_2}")
)
comparison = comparison.fillna(0)
comparison["abs_diff"] = (
    comparison[f"mean_abs_shap_{TARGET_1}"] - comparison[f"mean_abs_shap_{TARGET_2}"]
)
comparison["rank_diff"] = (
    comparison[f"rank_{TARGET_2}"] - comparison[f"rank_{TARGET_1}"]
)
comparison = comparison.sort_values(
    by=[f"mean_abs_shap_{TARGET_1}", f"mean_abs_shap_{TARGET_2}"],
    ascending=False
).reset_index(drop=True)
 
comparison.to_csv(f"{OUT_DIR}/rq2_comparative_shap_table.csv", index=False)
print(f"\n✅ Saved full comparison table: {OUT_DIR}/rq2_comparative_shap_table.csv")
 
# ═══════════════════════════════════════════════════════════════════════════
# 4. RANK CORRELATION — how similar/different are the two importance orderings?
# ═══════════════════════════════════════════════════════════════════════════
# Two versions are computed:
#   (a) FULL — across all features, including low-importance ones. This can
#       be misleading: if most features are near-zero "noise floor" for both
#       targets, their similarly-low ranks can dominate the correlation and
#       mask genuine divergence among the handful of features that actually
#       matter.
#   (b) TOP-N — restricted to only the N most important features (by combined
#       importance across both targets). This is the more informative measure
#       of whether the KEY drivers overlap, since it excludes noise-floor
#       agreement that isn't analytically meaningful.
 
TOP_N_FOR_CORRELATION = 15  # restrict the "meaningful" correlation to this many features
 
common = comparison[
    (comparison[f"rank_{TARGET_1}"] > 0) & (comparison[f"rank_{TARGET_2}"] > 0)
]
 
# (a) Full correlation — all features
rho_full, p_full = spearmanr(common[f"rank_{TARGET_1}"], common[f"rank_{TARGET_2}"])
 
# (b) Top-N correlation — only the most important features overall
common_ranked_by_importance = common.copy()
common_ranked_by_importance["combined_importance"] = (
    common_ranked_by_importance[f"mean_abs_shap_{TARGET_1}"]
    + common_ranked_by_importance[f"mean_abs_shap_{TARGET_2}"]
)
top_n_subset = common_ranked_by_importance.nlargest(TOP_N_FOR_CORRELATION, "combined_importance")
rho_topn, p_topn = spearmanr(top_n_subset[f"rank_{TARGET_1}"], top_n_subset[f"rank_{TARGET_2}"])
 
def interpret_rho(rho, p_value=None, n_features=None):
    """Interprets a Spearman correlation, correctly accounting for DIRECTION
    (positive = similar rankings, negative = inverted rankings) — not just
    magnitude. A previous version of this function only checked abs(rho),
    which mislabelled strong negative correlations (inverted rankings,
    evidence of DIVERGENCE) as if they showed similarity."""
    strength = "weak" if abs(rho) < 0.3 else "moderate" if abs(rho) < 0.6 else "strong"
 
    if rho >= 0:
        meaning = "the two models rank features similarly"
    else:
        meaning = "the two models rank features in close to opposite order — evidence of divergence, not similarity"
 
    sig_note = ""
    if p_value is not None and p_value >= 0.05:
        n_note = f" (n = {n_features})" if n_features else ""
        sig_note = (f"; note this did not reach conventional statistical significance "
                    f"(p = {p_value:.3f}){n_note}, a limitation of correlating a small number of ranks")
 
    return f"{strength} ({meaning}{sig_note})"
 
 
def rho_direction_word(rho):
    """Returns whether a correlation indicates similarity or divergence,
    for use in sentence templates — explicitly direction-aware."""
    if rho >= 0.6:
        return "closely resemble"
    elif rho >= 0.3:
        return "partially overlap with"
    elif rho > -0.3:
        return "differ from"
    elif rho > -0.6:
        return "diverge from, showing partial rank inversion relative to"
    else:
        return "are nearly inverted relative to"
 
 
def strength_word(rho):
    """Magnitude-only strength label (weak/moderate/strong), computed
    dynamically from the actual value rather than hardcoded in templates —
    a previous version of this script hardcoded 'strong' in one sentence
    template regardless of the actual correlation magnitude."""
    return "strong" if abs(rho) >= 0.6 else "moderate" if abs(rho) >= 0.3 else "weak"
 
interp_full = interpret_rho(rho_full, p_full, len(common))
interp_topn = interpret_rho(rho_topn, p_topn, TOP_N_FOR_CORRELATION)
 
print("\n" + "=" * 80)
print("RANK CORRELATION BETWEEN THE TWO FEATURE IMPORTANCE PROFILES")
print("=" * 80)
print(f"(a) FULL (all {len(common)} common features):")
print(f"    Spearman's rho = {rho_full:.3f}  (p = {'< .001' if p_full < 0.001 else round(p_full, 4)})")
print(f"    Interpretation: {interp_full}")
print(f"\n(b) TOP {TOP_N_FOR_CORRELATION} MOST IMPORTANT FEATURES ONLY:")
print(f"    Spearman's rho = {rho_topn:.3f}  (p = {'< .001' if p_topn < 0.001 else round(p_topn, 4)})")
print(f"    Interpretation: {interp_topn}")
print(f"    Features included: {', '.join(top_n_subset['feature'].tolist())}")
 
if abs(rho_full - rho_topn) > 0.3 or (rho_full >= 0) != (rho_topn >= 0):
    print(f"\n⚠ NOTE: the full-feature and top-{TOP_N_FOR_CORRELATION} correlations diverge substantially "
          f"({rho_full:.3f} vs {rho_topn:.3f}"
          f"{', and even flip sign' if (rho_full >= 0) != (rho_topn >= 0) else ''}). "
          f"This suggests agreement among low-importance/noise-floor features is masking genuine "
          f"divergence among the features that actually matter — report the top-N result as the "
          f"more analytically meaningful figure.")
 
print("\n--- Suggested dissertation text (Findings, RQ2) ---")
topn_sig_clause = (
    f", though this did not reach conventional statistical significance "
    f"(p = {p_topn:.3f}), a limitation of correlating a small number of ranks (n = {TOP_N_FOR_CORRELATION})"
    if p_topn >= 0.05 else f" (p = {'< .001' if p_topn < 0.001 else f'p = {round(p_topn,4)}'})"
)
 
if abs(rho_full - rho_topn) > 0.3 or (rho_full >= 0) != (rho_topn >= 0):
    if rho_topn < 0:
        print(
            f"'A Spearman rank correlation across all {len(common)} features suggested a "
            f"{strength_word(rho_full)} positive relationship (rho = {rho_full:.3f}, "
            f"{'p < .001' if p_full < 0.001 else f'p = {round(p_full,4)}'}); "
            f"however, this was driven by consistent ordering among a large number of low-importance features "
            f"common to both models, rather than agreement on the features that substantively drive each "
            f"outcome. Restricting the comparison to the {TOP_N_FOR_CORRELATION} most influential features "
            f"instead revealed a {strength_word(rho_topn)} NEGATIVE correlation (rho = {rho_topn:.3f}{topn_sig_clause}) — "
            f"indicating that features important to one outcome tend to be comparatively unimportant to the "
            f"other, and vice versa. This is consistent with the feature-level divergence shown directly in "
            f"the comparison table below, and provides clear evidence that {TARGET_1_LABEL.lower()} and "
            f"{TARGET_2_LABEL.lower()} are driven by substantially different adoption factors, directly "
            f"addressing RQ2.'"
        )
    else:
        print(
            f"'While a Spearman rank correlation across all {len(common)} features suggests a "
            f"{strength_word(rho_full)} "
            f"relationship (rho = {rho_full:.3f}, {'p < .001' if p_full < 0.001 else f'p = {round(p_full,4)}'}), "
            f"this is driven largely by agreement among low-importance features common to both models. "
            f"Restricting the comparison to the {TOP_N_FOR_CORRELATION} most influential features reveals a "
            f"markedly weaker relationship (rho = {rho_topn:.3f}{topn_sig_clause}), indicating that the "
            f"features which substantively drive each outcome {rho_direction_word(rho_topn)} each other — "
            f"suggesting the primary adoption factors underlying {TARGET_1_LABEL.lower()} and "
            f"{TARGET_2_LABEL.lower()} are largely distinct.'"
        )
else:
    print(
        f"'A Spearman rank correlation of rho = {rho_topn:.3f}{topn_sig_clause} among the "
        f"{TOP_N_FOR_CORRELATION} most influential features indicates that the adoption factors driving "
        f"{TARGET_1_LABEL.lower()} {rho_direction_word(rho_topn)} those driving {TARGET_2_LABEL.lower()}.'"
    )
 
# ═══════════════════════════════════════════════════════════════════════════
# 5. TOP DIVERGENT FEATURES — most interesting for Discussion
# ═══════════════════════════════════════════════════════════════════════════
 
comparison["divergence_score"] = (
    comparison[f"mean_abs_shap_{TARGET_1}"] - comparison[f"mean_abs_shap_{TARGET_2}"]
).abs()
top_divergent = comparison.sort_values("divergence_score", ascending=False).head(TOP_N_DIVERGENT)
 
print("\n" + "=" * 80)
print(f"TOP {TOP_N_DIVERGENT} MOST DIVERGENT FEATURES (matters a lot for one target, little for the other)")
print("=" * 80)
print(top_divergent[[
    "feature", f"mean_abs_shap_{TARGET_1}", f"mean_abs_shap_{TARGET_2}", "divergence_score"
]].to_string(index=False))
 
top_divergent.to_csv(f"{OUT_DIR}/rq2_top_divergent_features.csv", index=False)
print(f"\n✅ Saved: {OUT_DIR}/rq2_top_divergent_features.csv")
 
# ═══════════════════════════════════════════════════════════════════════════
# 6. TORNADO / DIVERGING BAR CHART — THE MAIN RQ2 VISUAL
# ═══════════════════════════════════════════════════════════════════════════
 
# Pick the features to display: union of top-N from each target's own ranking
top_features_1 = summary_1.head(TOP_N_DISPLAY)["feature"].tolist()
top_features_2 = summary_2.head(TOP_N_DISPLAY)["feature"].tolist()
display_features = list(dict.fromkeys(top_features_1 + top_features_2))  # preserves order, dedups
 
plot_df = comparison[comparison["feature"].isin(display_features)].copy()
# Order by combined importance for a clean visual
plot_df["combined"] = plot_df[f"mean_abs_shap_{TARGET_1}"] + plot_df[f"mean_abs_shap_{TARGET_2}"]
plot_df = plot_df.sort_values("combined", ascending=True)  # ascending for barh (top = largest)
 
fig, ax = plt.subplots(figsize=(11, max(6, len(plot_df) * 0.4)))
 
y_pos = np.arange(len(plot_df))
ax.barh(y_pos, -plot_df[f"mean_abs_shap_{TARGET_1}"], color=NAVY, label=TARGET_1_LABEL, height=0.65)
ax.barh(y_pos, plot_df[f"mean_abs_shap_{TARGET_2}"], color=AMBER, label=TARGET_2_LABEL, height=0.65)
 
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["feature"], fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
 
# Symmetric x-axis with absolute value labels
max_val = max(plot_df[f"mean_abs_shap_{TARGET_1}"].max(), plot_df[f"mean_abs_shap_{TARGET_2}"].max())
ax.set_xlim(-max_val * 1.15, max_val * 1.15)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.2f}" for x in xticks])
 
ax.set_xlabel("Mean |SHAP value|  (feature importance)", fontsize=10)
ax.set_title(
    f"RQ2 — Comparative SHAP Feature Importance\n{TARGET_1_LABEL} (left) vs {TARGET_2_LABEL} (right)",
    fontsize=13, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
 
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/rq2_comparative_shap_tornado.png", dpi=150, bbox_inches="tight")
print(f"\n✅ Saved main RQ2 figure: {OUT_DIR}/rq2_comparative_shap_tornado.png")
plt.close()
 
# ═══════════════════════════════════════════════════════════════════════════
# 7. SCATTER PLOT — importance in target 1 vs importance in target 2
# ═══════════════════════════════════════════════════════════════════════════
# A second, complementary view: if a feature matters equally for both, it
# sits near the diagonal. Points far off the diagonal are the divergent ones.
 
fig2, ax2 = plt.subplots(figsize=(8, 7))
ax2.scatter(
    comparison[f"mean_abs_shap_{TARGET_1}"],
    comparison[f"mean_abs_shap_{TARGET_2}"],
    alpha=0.6, color=NAVY, s=40, edgecolor="white", linewidth=0.5
)
 
# Label the most divergent + most important points
label_features = pd.concat([
    comparison.nlargest(5, f"mean_abs_shap_{TARGET_1}"),
    comparison.nlargest(5, f"mean_abs_shap_{TARGET_2}"),
]).drop_duplicates(subset="feature")
 
for _, row in label_features.iterrows():
    ax2.annotate(
        row["feature"],
        (row[f"mean_abs_shap_{TARGET_1}"], row[f"mean_abs_shap_{TARGET_2}"]),
        fontsize=7, alpha=0.85, xytext=(4, 4), textcoords="offset points"
    )
 
max_axis = max(comparison[f"mean_abs_shap_{TARGET_1}"].max(), comparison[f"mean_abs_shap_{TARGET_2}"].max()) * 1.1
ax2.plot([0, max_axis], [0, max_axis], color=GREY, linestyle="--", linewidth=1, label="Equal importance")
ax2.set_xlim(0, max_axis)
ax2.set_ylim(0, max_axis)
ax2.set_xlabel(f"Feature importance — {TARGET_1_LABEL}", fontsize=10)
ax2.set_ylabel(f"Feature importance — {TARGET_2_LABEL}", fontsize=10)
ax2.set_title(
    f"RQ2 — Feature Importance Agreement\n"
    f"Spearman ρ (top {TOP_N_FOR_CORRELATION}) = {rho_topn:.3f}  |  ρ (all features) = {rho_full:.3f}",
    fontsize=11, fontweight="bold"
)
ax2.legend(fontsize=9)
ax2.spines[["top", "right"]].set_visible(False)
 
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/rq2_comparative_shap_scatter.png", dpi=150, bbox_inches="tight")
print(f"✅ Saved supplementary scatter figure: {OUT_DIR}/rq2_comparative_shap_scatter.png")
plt.close()
 
# ═══════════════════════════════════════════════════════════════════════════
# DONE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("RQ2 COMPARATIVE SHAP ANALYSIS COMPLETE")
print("=" * 80)
print(f"""
Files generated in {OUT_DIR}/:
  rq2_comparative_shap_table.csv       — full merged comparison (all features)
  rq2_top_divergent_features.csv       — features that differ most between targets
  rq2_comparative_shap_tornado.png     — MAIN FIGURE: side-by-side importance bars
  rq2_comparative_shap_scatter.png     — supplementary: agreement scatter plot
 
Spearman rank correlation (all features):         rho = {rho_full:.3f} (p {'< .001' if p_full < 0.001 else f'= {round(p_full,4)}'})
Spearman rank correlation (top {TOP_N_FOR_CORRELATION} features):        rho = {rho_topn:.3f} (p {'< .001' if p_topn < 0.001 else f'= {round(p_topn,4)}'})
""")

In [ ]:
import pickle
import matplotlib.pyplot as plt
import shap

OUT_DIR = "pipeline_outputs"

# Load the saved SHAP objects for both targets
with open(f"{OUT_DIR}/shap/revenue_growth_percent_shap_values.pkl", "rb") as f:
    rev = pickle.load(f)

with open(f"{OUT_DIR}/shap/cost_reduction_percent_shap_values.pkl", "rb") as f:
    cost = pickle.load(f)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: productivity_change_percent vs its SHAP value (revenue growth model)
shap.dependence_plot(
    "productivity_change_percent",
    rev["shap_values"], rev["X_test_transformed"],
    feature_names=rev["feature_names"],
    ax=axes[0], show=False,
)
axes[0].set_title("Revenue Growth: productivity_change_percent", fontsize=11, fontweight="bold")

# Right: task_automation_rate vs its SHAP value (cost reduction model)
shap.dependence_plot(
    "task_automation_rate",
    cost["shap_values"], cost["X_test_transformed"],
    feature_names=cost["feature_names"],
    ax=axes[1], show=False,
)
axes[1].set_title("Cost Reduction: task_automation_rate", fontsize=11, fontweight="bold")

plt.suptitle("Figure 4.4: SHAP Dependence — Dominant Feature per Target", fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/comparative_shap/rq2_dependence_plots.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
df = pd.read_csv("pipeline_outputs/metrics/revenue_growth_percent_native_feature_importance.csv")
print(df.head(5))